In [ ]:
# Setup
import os

import numpy as np
import torch

from eval_refiner import get_test_dataloader, load_model
from srcs.datasets.vicocktail import load_vicocktail
from srcs.spm.spm_train import ensure_unigram
from srcs.spm.text_transofm import TextTransform
from test import (
    collect_predictions,
    create_length_matched_permutation,
    paired_bootstrap,
    tone_stripped_errors,
    word_errors,
    word_transitions,
)
from train_refiner import load_config

PROJECT_ROOT = os.getcwd()
CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.yaml")
CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "checkpoint-54908",
    "checkpoint-54908",
)
TEST_FRACTION = 1.0
BOOTSTRAP_SAMPLES = 10000
SEED = 42

config = load_config(CONFIG_PATH)
evaluation_config = config["evaluation"]
test_dataset = load_vicocktail(
    test_fraction=TEST_FRACTION, splits=("test",), seed=SEED
)["test"]
model_path, units_path = ensure_unigram()
text_transform = TextTransform(model_path, units_path)
permutation, length_differences = create_length_matched_permutation(test_dataset)
shuffled_dataset = test_dataset.select(permutation)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(text_transform.vocab_size, CHECKPOINT_PATH).to(device)
dataloader = get_test_dataloader(
    test_dataset, text_transform, evaluation_config
)
shuffled_dataloader = get_test_dataloader(
    shuffled_dataset, text_transform, evaluation_config
)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Device: {device}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Experiment 1: run inference and evaluate baseline, inner CTC, and final CTC
predictions = collect_predictions(
    model, dataloader, shuffled_dataloader, text_transform, device
)
references = predictions["references"]
baseline_errors, reference_words = word_errors(
    references, predictions["baseline"]
)
inner_errors, _ = word_errors(references, predictions["inner"])
final_errors, _ = word_errors(references, predictions["normal"])
word_count = reference_words.sum()
baseline_wer = baseline_errors.sum() / word_count
inner_wer = inner_errors.sum() / word_count
final_wer = final_errors.sum() / word_count

print(f"Baseline WER: {baseline_wer:.6f}")
print(f"Inner WER: {inner_wer:.6f}")
print(f"Final WER: {final_wer:.6f}")
print(f"Inner loss: {predictions['inner_loss']:.6f}")
print(f"Final loss: {predictions['final_loss']:.6f}")

In [ ]:
# Experiment 2: matched utterance errors and paired bootstrap
refiner_wins = int(np.sum(final_errors < baseline_errors))
baseline_wins = int(np.sum(baseline_errors < final_errors))
ties = int(np.sum(final_errors == baseline_errors))
baseline_bootstrap = paired_bootstrap(
    baseline_errors, final_errors, reference_words, BOOTSTRAP_SAMPLES, SEED
)

print(f"Refiner wins: {refiner_wins}")
print(f"Baseline wins: {baseline_wins}")
print(f"Ties: {ties}")
print(f"Final - baseline WER: {final_wer - baseline_wer:.6f}")
print(
    "Final - baseline bootstrap 95% CI: "
    f"[{baseline_bootstrap['lower']:.6f}, "
    f"{baseline_bootstrap['upper']:.6f}]"
)
print(
    "Final better than baseline proportion: "
    f"{baseline_bootstrap['improvement_probability']:.4f}"
)
print(
    "Significant improvement: "
    f"{'yes' if baseline_bootstrap['upper'] < 0.0 else 'no'}"
)

In [ ]:
# Experiment 3: word-level correction and damage transitions
transitions = word_transitions(
    references, predictions["baseline"], predictions["normal"]
)

print(f"Wrong -> Correct: {transitions['wrong_to_correct']}")
print(f"Correct -> Wrong: {transitions['correct_to_wrong']}")
print(f"Wrong -> Wrong: {transitions['wrong_to_wrong']}")
print(f"Correct -> Correct: {transitions['correct_to_correct']}")
print(f"Correction rate: {transitions['correction_rate'] * 100:.3f}%")
print(f"Damage rate: {transitions['damage_rate'] * 100:.3f}%")
print(f"Baseline insertions: {transitions['baseline_insertions']}")
print(f"Refiner insertions: {transitions['refiner_insertions']}")

In [ ]:
# Experiment 4: correct, zero, and length-matched shuffled visual evidence
shuffled_errors, _ = word_errors(references, predictions["shuffled"])
zero_errors, _ = word_errors(references, predictions["zero"])
shuffled_wer = shuffled_errors.sum() / word_count
zero_wer = zero_errors.sum() / word_count
visual_gain = zero_wer - final_wer
total_gain = baseline_wer - final_wer
visual_gain_ratio = (
    visual_gain / total_gain if total_gain != 0.0 else float("nan")
)

print(f"Baseline WER: {baseline_wer:.6f}")
print(f"Correct visual WER: {final_wer:.6f}")
print(f"Zero visual WER: {zero_wer:.6f}")
print(f"Shuffled visual WER: {shuffled_wer:.6f}")
print(f"Correct - zero WER: {final_wer - zero_wer:.6f}")
print(f"Correct - shuffled WER: {final_wer - shuffled_wer:.6f}")
print(f"Descriptive visual gain ratio: {visual_gain_ratio * 100:.3f}%")
print(
    "Shuffle length difference: "
    f"mean={length_differences.mean():.3f}, "
    f"max={length_differences.max()} frames"
)

In [ ]:
# Experiment 5: paired bootstrap for visual evidence
zero_bootstrap = paired_bootstrap(
    zero_errors, final_errors, reference_words, BOOTSTRAP_SAMPLES, SEED
)
shuffled_bootstrap = paired_bootstrap(
    shuffled_errors, final_errors, reference_words, BOOTSTRAP_SAMPLES, SEED
)

print(
    "Correct - zero bootstrap 95% CI: "
    f"[{zero_bootstrap['lower']:.6f}, {zero_bootstrap['upper']:.6f}]"
)
print(
    "Correct better than zero proportion: "
    f"{zero_bootstrap['improvement_probability']:.4f}"
)
print(
    "Correct better than zero: "
    f"{'yes' if zero_bootstrap['upper'] < 0.0 else 'no'}"
)
print(
    "Correct - shuffled bootstrap 95% CI: "
    f"[{shuffled_bootstrap['lower']:.6f}, "
    f"{shuffled_bootstrap['upper']:.6f}]"
)
print(
    "Correct better than shuffled proportion: "
    f"{shuffled_bootstrap['improvement_probability']:.4f}"
)
print(
    "Correct better than shuffled: "
    f"{'yes' if shuffled_bootstrap['upper'] < 0.0 else 'no'}"
)

In [ ]:
# Experiment 6: tone-stripped WER
baseline_tone_errors, _ = tone_stripped_errors(
    references, predictions["baseline"]
)
final_tone_errors, _ = tone_stripped_errors(
    references, predictions["normal"]
)
zero_tone_errors, _ = tone_stripped_errors(
    references, predictions["zero"]
)
shuffled_tone_errors, _ = tone_stripped_errors(
    references, predictions["shuffled"]
)
baseline_tone_wer = baseline_tone_errors.sum() / word_count
final_tone_wer = final_tone_errors.sum() / word_count
zero_tone_wer = zero_tone_errors.sum() / word_count
shuffled_tone_wer = shuffled_tone_errors.sum() / word_count
baseline_tone_fraction = (
    baseline_errors.sum() - baseline_tone_errors.sum()
) / baseline_errors.sum()
final_tone_fraction = (
    final_errors.sum() - final_tone_errors.sum()
) / final_errors.sum()

print(f"Baseline tone-stripped WER: {baseline_tone_wer:.6f}")
print(f"Correct visual tone-stripped WER: {final_tone_wer:.6f}")
print(f"Zero visual tone-stripped WER: {zero_tone_wer:.6f}")
print(f"Shuffled visual tone-stripped WER: {shuffled_tone_wer:.6f}")
print(
    "Baseline tone-stripping reduction: "
    f"{(baseline_wer - baseline_tone_wer) * 100:.3f} points"
)
print(
    "Correct visual tone-stripping reduction: "
    f"{(final_wer - final_tone_wer) * 100:.3f} points"
)
print(
    "Baseline errors removed by tone stripping: "
    f"{baseline_tone_fraction * 100:.3f}%"
)
print(
    "Correct visual errors removed by tone stripping: "
    f"{final_tone_fraction * 100:.3f}%"
)